In [0]:
# =============================================================================
# Gold · frota_eletrica_evolucao
# Evolução mensal da frota elétrica e híbrida por estado
# Granularidade: sigla_uf × nm_grupo × dt_referencia
# =============================================================================
 
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
%sql
USE CATALOG brazil_car_fleet;
USE SCHEMA gold;
DROP TABLE IF EXISTS gold.frota_evolucao;

In [0]:
# -----------------------------------------------------------------------------
# 1. Ler as tabelas Silver necessárias
# -----------------------------------------------------------------------------
 
df_fato  = spark.table("silver.fact_frota")
df_mun   = spark.table("silver.dim_municipio").select("id_municipio", "sigla_uf", "nm_uf", "nm_regiao")
df_comb  = spark.table("silver.dim_combustivel").select("id_combustivel", "nm_grupo")
df_data  = spark.table("silver.dim_data").select("id_data", "dt_referencia", "nr_mes", "nr_ano")

In [0]:
# -----------------------------------------------------------------------------
# 2. Join com as dimensões e filtrar apenas Elétrico Puro e Híbrido
# -----------------------------------------------------------------------------
 
df = (
    df_fato
    .join(df_mun,  on="id_municipio",   how="inner")
    .join(df_comb, on="id_combustivel", how="inner")
    .join(df_data, on="id_data",        how="inner")
    #.filter(F.col("nm_grupo").isin("Elétrico Puro", "Híbrido"))
)

In [0]:
# -----------------------------------------------------------------------------
# 3. Agregar: total por sigla_uf × nm_grupo × período
# -----------------------------------------------------------------------------
 
df = (
    df
    .groupBy("sigla_uf", "nm_uf", "nm_regiao", "nm_grupo", "id_data", "dt_referencia", "nr_mes", "nr_ano")
    .agg(F.sum("qtd_veiculos").cast("long").alias("qtd_veiculos"))
)
 

In [0]:
# -----------------------------------------------------------------------------
# 4. Total geral da frota por sigla_uf × período (para calcular % do total)
#    Usa toda a fato, sem filtro de grupo
# -----------------------------------------------------------------------------
 
df_total_frota = (
    df_fato
    .join(df_mun, on="id_municipio", how="inner")
    .groupBy("sigla_uf", "id_data")
    .agg(F.sum("qtd_veiculos").cast("long").alias("qtd_total_frota"))
)
 
df = df.join(df_total_frota, on=["sigla_uf", "id_data"], how="left")

In [0]:
# -----------------------------------------------------------------------------
# 5. Calcular % sobre o total da frota do estado
# -----------------------------------------------------------------------------
 
df = df.withColumn(
    "pct_frota_total",
    F.round(F.col("qtd_veiculos") / F.col("qtd_total_frota") * 100, 4)
)

In [0]:
# -----------------------------------------------------------------------------
# 6. Calcular variação MoM (Month over Month)
#    Janela: mesmo sigla_uf + nm_grupo, ordenado por id_data
# -----------------------------------------------------------------------------
 
w_uf_grupo = Window.partitionBy("sigla_uf", "nm_grupo").orderBy("id_data")
 
df = (
    df
    .withColumn("qtd_mes_anterior", F.lag("qtd_veiculos", 1).over(w_uf_grupo))
    .withColumn(
        "variacao_mom",
        F.round(
            (F.col("qtd_veiculos") - F.col("qtd_mes_anterior")) / F.col("qtd_mes_anterior") * 100,
            4
        )
    )
    .drop("qtd_mes_anterior")
)

In [0]:
# -----------------------------------------------------------------------------
# 7. Calcular variação YoY (Year over Year — mesmo mês do ano anterior)
#    Para 2024 ficará nulo — comportamento esperado
# -----------------------------------------------------------------------------
 
df_yoy = (
    df
    .select(
        "sigla_uf", "nm_grupo", "nr_mes",
        (F.col("nr_ano") + 1).alias("nr_ano"),        # shifta o ano +1 para fazer o join
        F.col("qtd_veiculos").alias("qtd_ano_anterior")
    )
)

In [0]:
df = (
    df
    .join(df_yoy, on=["sigla_uf", "nm_grupo", "nr_mes", "nr_ano"], how="left")
    .withColumn(
        "variacao_yoy",
        F.round(
            (F.col("qtd_veiculos") - F.col("qtd_ano_anterior")) / F.col("qtd_ano_anterior") * 100,
            4
        )
    )
    .drop("qtd_ano_anterior")
)

In [0]:
# -----------------------------------------------------------------------------
# 8. Selecionar e ordenar colunas finais
# -----------------------------------------------------------------------------
 
df_gold = (
    df
    .select(
        "sigla_uf",
        "nm_uf",
        "nm_regiao",
        "nm_grupo",
        "id_data",
        "dt_referencia",
        "nr_mes",
        "nr_ano",
        "qtd_veiculos",
        "qtd_total_frota",
        "pct_frota_total",
        "variacao_mom",
        "variacao_yoy",
    )
    .orderBy("sigla_uf", "nm_grupo", "id_data")
)

In [0]:
# -----------------------------------------------------------------------------
# 9. Auditoria rápida
# -----------------------------------------------------------------------------
 
print(f"Total de registros       : {df_gold.count():,}")
print(f"Estados cobertos         : {df_gold.select('sigla_uf').distinct().count()}")
print(f"Períodos cobertos        : {df_gold.select('id_data').distinct().count()}")
print(f"Registros com YoY nulo   : {df_gold.filter(F.col('variacao_yoy').isNull()).count():,}  (esperado para 2024)")
print(f"Registros com MoM nulo   : {df_gold.filter(F.col('variacao_mom').isNull()).count():,}  (esperado para o 1º mês de cada série)")
 
print("\nAmostra — SP, Elétrico Puro:")
(
    df_gold
    .filter((F.col("sigla_uf") == "SP") & (F.col("nm_grupo") == "Elétrico Puro"))
    .select("dt_referencia", "qtd_veiculos", "pct_frota_total", "variacao_mom", "variacao_yoy")
    .show(24, truncate=False)
)

In [0]:
# -----------------------------------------------------------------------------
# 10. Salvar na Gold como Delta, particionado por nr_ano
# -----------------------------------------------------------------------------
 
(
    df_gold
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("nr_ano")
    .saveAsTable("gold.frota_evolucao")
)
 
print("\ngold.frota_evolucao salva com sucesso.")